In [2]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

from mapdl_utils import start_mapdl
from run_single_case import run_single_case

ImportError: cannot import name 'setup_element_type' from 'solver' (d:\ANSYS仿真项目\APDL_cache_files\Python\ThermalStress_iter\solver.py)

In [ ]:
# 启动 MAPDL
mapdl = start_mapdl(nproc=16)

In [ ]:
# 先跑一个单工况测试
result = run_single_case(
    mapdl,
    user_params={"H_w": 0.4},
    outdir="results/test_case"
)

In [ ]:
# 查看关键指标
print(result["metrics"])

In [ ]:
# 查看中线数据
df_mid = result["df_mid"]
df_mid.head()

In [ ]:
# 参数扫描
hw_list = [0.2, 0.4, 0.6, 0.8, 1.0]
summary = []

for hw in hw_list:
    print(f"正在计算 H_w = {hw:.3f} mm")

    result = run_single_case(
        mapdl,
        user_params={"H_w": hw},
        outdir=f"results/tw_{hw:.3f}"
    )

    summary.append(result["metrics"])

df_summary = pd.DataFrame(summary)
df_summary

In [ ]:
# 保存汇总
os.makedirs("results", exist_ok=True)
df_summary.to_csv("results/summary.csv", index=False, encoding="utf-8-sig")
print("已保存 results/summary.csv")

In [ ]:
# 画扫描结果图
plt.figure(figsize=(7, 5))
plt.plot(df_summary["H_w_mm"], df_summary["W_S1_max"], "o-", label="W_S1_max")
plt.plot(df_summary["H_w_mm"], df_summary["W_SEQV_max"], "s-", label="W_SEQV_max")
plt.xlabel("H_w (mm)")
plt.ylabel("Stress (MPa)")
plt.title("Effect of W thickness on stress")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 关闭 MAPDL
mapdl.exit()